In [1]:
import os
import pandas as pd
from glob import glob
import numpy as np

os.chdir('/store/carroll/sbgplants/')

In [2]:
# file paths
raw = 'data/raw'
doi = os.path.join(raw, '10.15485.1618130') # Locations, metadata, and species cover from field sampling survey associated with NEON AOP survey, East River, CO 2018

out_folder = 'data/out_csv'

table = 'sample'

In [12]:
# load relevant output tables
campaign = pd.read_csv(os.path.join(out_folder, 'campaign.csv'))
plot_event_metadata = pd.read_csv(os.path.join(out_folder, 'insitu_plot_event.csv'))[['plot_name', 'insitu_plot_event_id']]

In [13]:
plot_event_metadata

,plot_name,insitu_plot_event_id
0,001-ER18,0
1,002-ER18,1
2,003-ER18,2
3,004-ER18,3
4,005-ER18,4
...,...,...
472,474-ER18,472
473,475-ER18,473
474,476-ER18,474
475,477-ER18,475


In [4]:
# load, update relevant raw tables

# fractional cover
fractional_cover = pd.read_csv(os.path.join(doi, 'fractional_cover.csv'))
# fix typos
fractional_cover.loc[fractional_cover.CoverCode=='engelmann', 'CoverCode'] = 'Engelmann'
fractional_cover.loc[fractional_cover.CoverCode=='RibMon', 'CoverCode'] = 'Gooseberry' # confirm this with Dana?
fractional_cover.loc[fractional_cover.CoverCode=='RubIda', 'CoverCode'] = 'Raspberry' # confirm this with Dana?
# merge duplicate rows
keys = ['CoverCode', 'SampleSiteCode']
agg_ = {
    'SamplingArea': 'first',
    'CollectionDate': 'first',
    'FractionalCover': 'sum',
    'Note': 'first'
}
fractional_cover = (
    fractional_cover
    .groupby(keys, dropna=False, sort=False)
    .agg(agg_)
    .reset_index()
)

# get species_or_type from spp list
species_list = pd.read_csv(os.path.join(doi, 'species_list.csv'))
species_list['species_or_type'] = species_list['Genus'] + ' ' + species_list['Species']
species_list.loc[species_list['species_or_type'].isna(), 'species_or_type'] = species_list.loc[species_list['Genus'].isna(), 'CoverCode']
species_list = species_list[['CoverCode', 'species_or_type']]

fractional_cover = pd.merge(fractional_cover, species_list, on='CoverCode', how='left', suffixes=('',''))
fractional_cover

,CoverCode,SampleSiteCode,SamplingArea,CollectionDate,FractionalCover,Note,species_or_type
0,Moss,276-ER18,RCK,6/23/2018,5,not in species list,Moss
1,PotPul,001-ER18,RM,6/14/2018,85,None,Potentilla pulcherrima
2,Litter,001-ER18,RM,6/14/2018,10,None,Litter
3,OF,001-ER18,RM,6/14/2018,5,None,OF
4,LupBak,002-ER18,RM,6/14/2018,30,None,Lupinus bakeri
...,...,...,...,...,...,...,...
1257,Lodgepole,474-ER18,GS,7/30/2018,100,None,Pinus contorta
1258,Lodgepole,475-ER18,GS,7/30/2018,100,None,Pinus contorta
1259,Lodgepole,476-ER18,GS,7/30/2018,100,None,Pinus contorta
1260,Engelmann,477-ER18,GS,7/30/2018,100,None,Picea engelmannii


In [23]:
# prepare & populate out table
out_table = pd.DataFrame(index=range(len(fractional_cover)))

out_table['sample_name'] = fractional_cover['SampleSiteCode'] + '_' + fractional_cover['species_or_type']
out_table['sample_name'] = out_table['sample_name'].str.replace(' ', '', regex=False)

out_table['plot_name'] = fractional_cover['SampleSiteCode']
out_table = out_table.merge(plot_event_metadata, on='plot_name', how='left')
out_table = out_table.drop(columns='plot_name')

out_table['species_or_type'] = fractional_cover['species_or_type']
out_table['phenophase'] = None
out_table['sample_fc_class'] = None

# map fractional class
fc_class = {
    'Bare': 'soil',
    'Litter': 'npv',
    # 'Moss': 'npv' # Dana said moss "might be pv"
}
out_table['sample_fc_class'] = out_table['species_or_type'].map(fc_class).fillna('pv')
out_table['sample_fc_percent'] = fractional_cover['FractionalCover']

out_table

,sample_name,insitu_plot_event_id,species_or_type,phenophase,sample_fc_class,sample_fc_percent
0,276-ER18_Moss,275,Moss,None,pv,5
1,001-ER18_Potentillapulcherrima,0,Potentilla pulcherrima,None,pv,85
2,001-ER18_Litter,0,Litter,None,npv,10
3,001-ER18_OF,0,OF,None,pv,5
4,002-ER18_Lupinusbakeri,1,Lupinus bakeri,None,pv,30
...,...,...,...,...,...,...
1257,474-ER18_Pinuscontorta,472,Pinus contorta,None,pv,100
1258,475-ER18_Pinuscontorta,473,Pinus contorta,None,pv,100
1259,476-ER18_Pinuscontorta,474,Pinus contorta,None,pv,100
1260,477-ER18_Piceaengelmannii,475,Picea engelmannii,None,pv,100


In [25]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)